In [20]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
import operator

In [21]:
model = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash')

In [22]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal['approved', 'needs_improvement']
    feedback: str
    iterations: int
    max_iteration: int
    
    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [23]:
def gen_tweet(state: TweetState) -> TweetState:
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    # send generator_llm
    response = model.invoke(messages).content

    # return response
    return {'tweet': response, 'tweet_history': [response]}

In [24]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [25]:
eval_llm = model.with_structured_output(TweetEvaluation)

In [26]:
def eval_tweets(state: TweetState) -> TweetState:
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response = eval_llm.invoke(messages)

    return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [27]:
def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = model.invoke(messages).content
    iteration = state['iterations'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [28]:
def route_evaluation(state: TweetState):

    if state['evaluation'] == 'approved' or state['iterations'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [29]:
graph = StateGraph(TweetState)

graph.add_node('generate', gen_tweet)
graph.add_node('evaluate', eval_tweets)
graph.add_node('optimise', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimise'})

graph.add_edge('optimise', 'evaluate')

workflow = graph.compile()

In [32]:
initial_state = {
    "topic": "srhberhb",
    "iterations": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

In [33]:
result

{'topic': 'srhberhb',
 'tweet': 'Trying to explain my life choices after 2 hours of doomscrolling. All that came out was "srhberhb." Checks out.',
 'evaluation': 'approved',
 'feedback': "This tweet effectively captures a relatable modern experience with a concise and humorous delivery. The gibberish 'srhberhb' is a strong, original punchline that enhances the humor and makes it feel fresh, rather than a rehashed observation. It's short, punchy, and highly shareable due to its universal relatability for anyone who has experienced the brain-fog of doomscrolling, making its virality potential high. The format is impeccable, avoiding common pitfalls like Q&A or traditional setup-punchline structures, and the concluding 'Checks out' provides a dry, self-aware finish that strengthens the joke rather than deflating it.",
 'iterations': 1,
 'max_iteration': 5,
 'tweet_history': ['Trying to explain my life choices after 2 hours of doomscrolling. All that came out was "srhberhb." Checks out.'],